### Note:
##### 1. Completed Part 1 within one day
##### 2. Part 2 was not attempted due to time constraints
##### 3. Lots of print statements and table views for ease of reviewing. Can be commented out in production
##### 4. Given more time, the notebook could be further enhanced with more comprehensive documentation, expanded exception handling, configuration externalisation, automated testing, and structured logging
##### 5. The solution emphasises readability, auditability, and maintainability by separating ingestion, validation, transformation, governance, and output generation into distinct pipeline stages

# Import libraries

In [1]:
!pip install great_expectations


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import requests
import pandas as pd
import logging
import time

from IPython.display import JSON, display, clear_output
from datetime import date, datetime, timezone
from typing import Any
from dateutil.relativedelta import relativedelta
from pathlib import Path

# Bronze Layer

### Bronze 1: Retrieve raw metadata

In [2]:
# =============================================================================
# Retrieve the metadata catalogue for the HDB resale flat collection.
# Dyamically and automatically extracts list of existing and new (future) child datasets without code change.
# =============================================================================

COLLECTION_ID = 189

METADATA_URL = (
    f"https://api-production.data.gov.sg/v2/public/api/collections/"
    f"{COLLECTION_ID}/metadata"
)

response = requests.get(
    METADATA_URL,
    timeout=30
)

response.raise_for_status()

metadata = response.json()

print(
    f"Metadata successfully retrieved for collection {COLLECTION_ID}."
)

print(
    f"Child datasets discovered: "
    f"{len(metadata['data']['collectionMetadata']['childDatasets'])}"
)

# Display the metadata for inspection
JSON(metadata)

Metadata successfully retrieved for collection 189.
Child datasets discovered: 5


<IPython.core.display.JSON object>

### Bronze 2: Extract raw child datasets

In [4]:
# =============================================================================
# Purpose:
# 1. Discover every child dataset from the collection metadata.
# 2. Extract every row through paginated API requests.
# 3. Preserve each child dataset separately for schema validation.
# 4. Create an extraction audit table for reconciliation.
#
# No cleaning, renaming, type conversion or deduplication is performed here.
# =============================================================================

API_BASE_URL = (
    "https://api-production.data.gov.sg/v2/public/api/datasets"
)

PAGE_LIMIT = 2_000
MAX_RETRIES = 3
TIMEOUT_SECONDS = 30

# Set to True only when pagination details are required for debugging.
SHOW_PAGE_PROGRESS = False


def extract_child_dataset(
    dataset_id: str,
    dataset_number: int,
    total_datasets: int
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Extract one complete child dataset using API pagination.

    Returns:
        child_df:
            Raw child dataset stored as a pandas DataFrame.

        audit_record:
            Extraction metadata used for reconciliation and auditing.
    """

    dataset_url = (
        f"{API_BASE_URL}/{dataset_id}/list-rows"
    )

    extraction_started_at = datetime.now(timezone.utc)

    offset = 0
    extracted_rows: list[dict[str, Any]] = []
    pages_extracted = 0

    print("\n" + "=" * 90)
    print(f"EXTRACTING CHILD DATASET: {dataset_id}")
    print("=" * 90)

    with requests.Session() as session:

        while True:
            
            if not SHOW_PAGE_PROGRESS:
            
                clear_output(wait=True)
            
                dots = "." * ((pages_extracted % 3) + 1)
            
                print(
                    f"Extracting child dataset "
                    f"{dataset_number} of {total_datasets}{dots}"
                )
                print(f"Dataset ID : {dataset_id}")
                print(f"Pages      : {pages_extracted}")
                print(f"Records    : {len(extracted_rows):,}")
    
            # Parameters used for the current paginated API request.
            request_parameters = {
                "limit": PAGE_LIMIT,
                "offset": offset
            }

            response_payload = None

            for attempt_number in range(1, MAX_RETRIES + 1):

                try:

                    response = session.get(
                        dataset_url,
                        params=request_parameters,
                        timeout=TIMEOUT_SECONDS
                    )

                    response.raise_for_status()

                    response_payload = response.json()

                    break

                except (
                    requests.RequestException,
                    ValueError
                ) as error:

                    print(
                        f"Attempt {attempt_number}/{MAX_RETRIES} failed "
                        f"for dataset {dataset_id} at offset {offset:,}."
                    )

                    print(f"Error: {error}")

                    if attempt_number == MAX_RETRIES:
                        raise RuntimeError(
                            f"Extraction failed for child dataset "
                            f"{dataset_id} at offset {offset:,}."
                        ) from error

                    retry_delay = attempt_number * 2

                    print(
                        f"Retrying in {retry_delay} seconds..."
                    )

                    time.sleep(retry_delay)

            rows = (
                response_payload
                .get("data", {})
                .get("rows", [])
            )

            if not rows:
                break

            # Add technical lineage fields without changing source values.
            for row in rows:
                row["_dataset_id"] = dataset_id

            extracted_rows.extend(rows)

            pages_extracted += 1

            # Pagination details are hidden by default.
            # Set SHOW_PAGE_PROGRESS = True for debugging.
            if SHOW_PAGE_PROGRESS:
                print(
                    f"Page {pages_extracted:>3} | "
                    f"Offset {offset:>8,} | "
                    f"Rows received {len(rows):>5,} | "
                    f"Running total {len(extracted_rows):>9,}"
                )

            # A page containing fewer records than PAGE_LIMIT is the last page.
            if len(rows) < PAGE_LIMIT:
                break

            offset += PAGE_LIMIT

    extraction_completed_at = datetime.now(timezone.utc)

    child_df = pd.DataFrame(extracted_rows)

    audit_record = {
        "dataset_id": dataset_id,
        "extraction_status": "SUCCESS",
        "row_count": len(child_df),
        "column_count": len(child_df.columns),
        "page_count": pages_extracted,
        "minimum_month": (
            child_df["month"].min()
            if "month" in child_df.columns and not child_df.empty
            else None
        ),
        "maximum_month": (
            child_df["month"].max()
            if "month" in child_df.columns and not child_df.empty
            else None
        ),
        "column_names": list(child_df.columns),
        "extraction_started_utc": extraction_started_at.isoformat(),
        "extraction_completed_utc": extraction_completed_at.isoformat()
    }

    print("-" * 90)
    print(f"Dataset ID    : {dataset_id}")
    print(f"Total Rows    : {len(child_df):,}")
    print(f"Total Columns : {len(child_df.columns)}")
    print(f"Pages Read    : {pages_extracted:,}")

    if "month" in child_df.columns and not child_df.empty:
        print(
            f"Date Range    : "
            f"{child_df['month'].min()} to "
            f"{child_df['month'].max()}"
        )

    print(f"Columns       : {list(child_df.columns)}")

    if not child_df.empty:

        print("\nRaw Preview — First 3 Records\n")

        preview_columns = [
            column
            for column in [
                "_dataset_id",
                "month",
                "town",
                "flat_type",
                "block",
                "street_name",
                "floor_area_sqm",
                "resale_price"
            ]
            if column in child_df.columns
        ]

        display(
            child_df[preview_columns].head(3)
        )

    return child_df, audit_record


# =============================================================================
# DISCOVER CHILD DATASETS FROM METADATA
# =============================================================================

child_dataset_ids = (
    metadata
    .get("data", {})
    .get("collectionMetadata", {})
    .get("childDatasets", [])
)

if not child_dataset_ids:
    raise ValueError(
        "No child datasets were found in the collection metadata."
    )

print("\n" + "=" * 90)
print("BRONZE EXTRACTION STARTED")
print("=" * 90)

print(f"Collection ID             : {COLLECTION_ID}")
print(f"Child datasets discovered : {len(child_dataset_ids)}")
print(f"Dataset IDs               : {child_dataset_ids}")

# =============================================================================
# INITIALISE BRONZE EXTRACTION CONTAINERS
# =============================================================================

bronze_child_datasets: dict[str, pd.DataFrame] = {}

bronze_extraction_audit: list[dict[str, Any]] = []


# =============================================================================
# EXTRACT AND STORE EACH CHILD DATASET
# =============================================================================

total_child_datasets = len(child_dataset_ids)

for dataset_number, dataset_id in enumerate(
    child_dataset_ids,
    start=1
):

    child_df, audit_record = extract_child_dataset(
        dataset_id=dataset_id,
        dataset_number=dataset_number,
        total_datasets=total_child_datasets
    )

    bronze_child_datasets[dataset_id] = child_df
    bronze_extraction_audit.append(audit_record)

    # Keep the completed child-dataset summary visible briefly.
    time.sleep(5)

    # Clear the final child-dataset preview before showing the audit table.
    if dataset_number == total_child_datasets:
        clear_output(wait=True)
        
# =============================================================================
# CREATE BRONZE EXTRACTION AUDIT TABLE
# =============================================================================

bronze_extraction_audit_df = pd.DataFrame(
    bronze_extraction_audit
)

total_bronze_records = sum(
    len(child_df)
    for child_df in bronze_child_datasets.values()
)

print("\n" + "=" * 90)
print("BRONZE EXTRACTION AUDIT SUMMARY")
print("=" * 90)

display(
    bronze_extraction_audit_df[
        [
            "dataset_id",
            "extraction_status",
            "row_count",
            "column_count",
            "page_count",
            "minimum_month",
            "maximum_month"
        ]
    ]
)

print(f"\nChild Datasets Extracted : {len(bronze_child_datasets)}")
print(f"Total Bronze Records     : {total_bronze_records:,}")

all_extractions_successful = (
    len(bronze_child_datasets) == len(child_dataset_ids)
    and
    bronze_extraction_audit_df[
        "extraction_status"
    ].eq("SUCCESS").all()
)

if all_extractions_successful:
    print("Extraction Validation     : PASS ✓")
else:
    print("Extraction Validation     : FAIL ✗")

    raise RuntimeError(
        "One or more child datasets were not extracted successfully."
    )


BRONZE EXTRACTION AUDIT SUMMARY


,dataset_id,extraction_status,row_count,column_count,page_count,minimum_month,maximum_month
0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,SUCCESS,235808,13,118,2017-01,2026-07
1,d_43f493c6c50d54243cc1eab0df142d6a,SUCCESS,369651,12,185,2000-01,2012-02
2,d_2d5ff9ea31397b66239f245f57751537,SUCCESS,52203,12,27,2012-03,2014-12
3,d_ebc5ab87086db484f88045b47411ebc5,SUCCESS,287196,12,144,1990-01,1999-12
4,d_ea9ed51da2787afaf8e51f827c304208,SUCCESS,37153,13,19,2015-01,2016-12



Child Datasets Extracted : 5
Total Bronze Records     : 982,011
Extraction Validation     : PASS ✓


### Bronze 3: Audit raw schema

In [5]:
# Collect every unique column name found across all child datasets.
# Compares columns by name, regardless of where each column appears
all_column_names = sorted(
    {
        column
        for child_df in bronze_child_datasets.values()
        for column in child_df.columns
    }
)

schema_presence_records = []

dataset_labels = {
    dataset_id: f"Child_{position}"
    for position, dataset_id in enumerate(
        bronze_child_datasets.keys(),
        start=1
    )
}

for column_name in all_column_names:

    comparison_record = {
        "column_name": column_name
    }

    for dataset_id, child_df in bronze_child_datasets.items():

        dataset_label = dataset_labels[dataset_id]

        comparison_record[dataset_label] = (
            "✓"
            if column_name in child_df.columns
            else "MISSING"
        )

    schema_presence_records.append(
        comparison_record
    )

schema_presence_df = pd.DataFrame(
    schema_presence_records
)

print("\n" + "=" * 90)
print("COLUMN PRESENCE MATRIX")
print("=" * 90)

display(schema_presence_df)


COLUMN PRESENCE MATRIX


,column_name,Child_1,Child_2,Child_3,Child_4,Child_5
0,_dataset_id,✓,✓,✓,✓,✓
1,block,✓,✓,✓,✓,✓
2,flat_model,✓,✓,✓,✓,✓
3,flat_type,✓,✓,✓,✓,✓
4,floor_area_sqm,✓,✓,✓,✓,✓
5,lease_commence_date,✓,✓,✓,✓,✓
6,month,✓,✓,✓,✓,✓
7,remaining_lease,✓,MISSING,MISSING,MISSING,✓
8,resale_price,✓,✓,✓,✓,✓
9,storey_range,✓,✓,✓,✓,✓


In [6]:
# =============================================================================
# DYNAMIC COLUMN PRESENCE AND NAME CONSISTENCY MATRIX
# =============================================================================
# Compare schemas by column name
#
# This identifies:
# - columns present in every child dataset
# - columns missing from some child datasets
# - differences caused only by capitalisation, spaces, hyphens or underscores
# - possible true column-name mismatches

# =============================================================================

import re
import pandas as pd
from IPython.display import display


def normalise_column_name(column_name: str) -> str:
    """
    Normalise a column name for comparison only.

    Examples:
        'Flat Type'     -> 'flat_type'
        'FLAT_TYPE'     -> 'flat_type'
        ' flat-type '   -> 'flat_type'
        'flat__type'    -> 'flat_type'
    """

    normalised_name = str(column_name).strip().lower()

    normalised_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        normalised_name
    )

    return normalised_name.strip("_")


# Create dynamic child labels.
dataset_labels = {
    dataset_id: f"Child_{position}"
    for position, dataset_id in enumerate(
        bronze_child_datasets.keys(),
        start=1
    )
}

dataset_count = len(bronze_child_datasets)


# =============================================================================
# BUILD A NORMALISED COLUMN CATALOGUE
# =============================================================================

normalised_column_catalogue = {}

for dataset_id, child_df in bronze_child_datasets.items():

    for original_column_name in child_df.columns:

        normalised_name = normalise_column_name(
            original_column_name
        )

        if normalised_name not in normalised_column_catalogue:
            normalised_column_catalogue[normalised_name] = {}

        normalised_column_catalogue[normalised_name][dataset_id] = (
            original_column_name
        )


# =============================================================================
# BUILD THE COMPARISON MATRIX
# =============================================================================

schema_comparison_records = []

for normalised_name in sorted(normalised_column_catalogue):

    names_by_dataset = normalised_column_catalogue[
        normalised_name
    ]

    comparison_record = {
        "normalised_column_name": normalised_name
    }

    original_names_present = []

    for dataset_id in bronze_child_datasets:

        dataset_label = dataset_labels[dataset_id]

        if dataset_id in names_by_dataset:

            original_name = names_by_dataset[dataset_id]

            comparison_record[dataset_label] = original_name
            original_names_present.append(original_name)

        else:

            comparison_record[dataset_label] = "MISSING"

    # -------------------------------------------------------------------------
    # Determine the row-level consistency result
    # -------------------------------------------------------------------------

    present_count = len(original_names_present)
    unique_original_names = set(original_names_present)

    if present_count < dataset_count:

        comparison_record["schema_consistency"] = (
            f"REVIEW - Missing from "
            f"{dataset_count - present_count} dataset(s)"
        )

    elif len(unique_original_names) == 1:

        comparison_record["schema_consistency"] = (
            "PASS - Exact Match"
        )

    else:

        comparison_record["schema_consistency"] = (
            "WARNING - Formatting Difference"
        )

    schema_comparison_records.append(
        comparison_record
    )


schema_comparison_df = pd.DataFrame(
    schema_comparison_records
)


# =============================================================================
# DISPLAY FULL SCHEMA COMPARISON
# =============================================================================

print("\n" + "=" * 110)
print("COLUMN PRESENCE AND NAME CONSISTENCY MATRIX")
print("=" * 110)

display(schema_comparison_df)


COLUMN PRESENCE AND NAME CONSISTENCY MATRIX


,normalised_column_name,Child_1,Child_2,Child_3,Child_4,Child_5,schema_consistency
0,block,block,block,block,block,block,PASS - Exact Match
1,dataset_id,_dataset_id,_dataset_id,_dataset_id,_dataset_id,_dataset_id,PASS - Exact Match
2,flat_model,flat_model,flat_model,flat_model,flat_model,flat_model,PASS - Exact Match
3,flat_type,flat_type,flat_type,flat_type,flat_type,flat_type,PASS - Exact Match
4,floor_area_sqm,floor_area_sqm,floor_area_sqm,floor_area_sqm,floor_area_sqm,floor_area_sqm,PASS - Exact Match
5,lease_commence_date,lease_commence_date,lease_commence_date,lease_commence_date,lease_commence_date,lease_commence_date,PASS - Exact Match
6,month,month,month,month,month,month,PASS - Exact Match
7,remaining_lease,remaining_lease,MISSING,MISSING,MISSING,remaining_lease,REVIEW - Missing from 3 dataset(s)
8,resale_price,resale_price,resale_price,resale_price,resale_price,resale_price,PASS - Exact Match
9,storey_range,storey_range,storey_range,storey_range,storey_range,storey_range,PASS - Exact Match


### Bronze 4: Raw schema validation gate

In [7]:
# Purpose:
# Evaluate the raw child dataset schemas before alignment and concatenation.
#
# Rules:
# 1. PASS - Exact Match:
#       Allow the pipeline to proceed.
#
# 2. REVIEW - Missing from ... dataset(s):
#       Allow the pipeline to proceed.
#       Missing columns will later be added as null values.
#       Log a non-blocking alert for audit and demonstration purposes.
#
# 3. WARNING - Formatting Difference:
#       Stop the pipeline for human review.
#
# 4. Any unknown schema status:
#       Stop the pipeline because the condition has not been approved.
# =============================================================================

import logging

from IPython.display import display


# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True
)

bronze_schema_logger = logging.getLogger(
    "bronze_schema_validation_gate"
)


# =============================================================================
# VALIDATE REQUIRED INPUT
# =============================================================================

required_schema_columns = {
    "normalised_column_name",
    "schema_consistency"
}

missing_required_schema_columns = (
    required_schema_columns
    - set(schema_comparison_df.columns)
)

if missing_required_schema_columns:
    raise KeyError(
        "schema_comparison_df is missing the required columns: "
        f"{sorted(missing_required_schema_columns)}"
    )


# =============================================================================
# CLASSIFY SCHEMA VALIDATION RESULTS
# =============================================================================

exact_match_mask = (
    schema_comparison_df["schema_consistency"]
    .eq("PASS - Exact Match")
)

missing_column_mask = (
    schema_comparison_df["schema_consistency"]
    .str.startswith(
        "REVIEW - Missing from",
        na=False
    )
)

formatting_difference_mask = (
    schema_comparison_df["schema_consistency"]
    .eq("WARNING - Formatting Difference")
)

recognised_status_mask = (
    exact_match_mask
    | missing_column_mask
    | formatting_difference_mask
)

unknown_status_mask = ~recognised_status_mask


# Store the affected column names for later use and auditing.
exact_match_columns = (
    schema_comparison_df.loc[
        exact_match_mask,
        "normalised_column_name"
    ]
    .tolist()
)

missing_columns = (
    schema_comparison_df.loc[
        missing_column_mask,
        "normalised_column_name"
    ]
    .tolist()
)

formatting_difference_columns = (
    schema_comparison_df.loc[
        formatting_difference_mask,
        "normalised_column_name"
    ]
    .tolist()
)

unknown_status_columns = (
    schema_comparison_df.loc[
        unknown_status_mask,
        "normalised_column_name"
    ]
    .tolist()
)


# =============================================================================
# CREATE SCHEMA-GATE AUDIT SUMMARY
# =============================================================================

bronze_schema_gate_audit = {
    "total_columns_reviewed": len(schema_comparison_df),
    "exact_match_count": len(exact_match_columns),
    "missing_column_count": len(missing_columns),
    "formatting_difference_count": len(
        formatting_difference_columns
    ),
    "unknown_status_count": len(unknown_status_columns),
    "exact_match_columns": exact_match_columns,
    "missing_columns": missing_columns,
    "formatting_difference_columns": (
        formatting_difference_columns
    ),
    "unknown_status_columns": unknown_status_columns
}


# =============================================================================
# PRINT SCHEMA-GATE SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("BRONZE SCHEMA VALIDATION GATE")
print("=" * 100)

print(
    f"Total columns reviewed            : "
    f"{bronze_schema_gate_audit['total_columns_reviewed']}"
)

print(
    f"Exact-match columns               : "
    f"{bronze_schema_gate_audit['exact_match_count']}"
)

print(
    f"Columns missing from some sources : "
    f"{bronze_schema_gate_audit['missing_column_count']}"
)

print(
    f"Formatting differences detected   : "
    f"{bronze_schema_gate_audit['formatting_difference_count']}"
)

print(
    f"Unknown schema statuses           : "
    f"{bronze_schema_gate_audit['unknown_status_count']}"
)


# =============================================================================
# BLOCKING CONDITION 1 - FORMATTING DIFFERENCES
# =============================================================================

if formatting_difference_columns:

    bronze_schema_logger.error(
        "PIPELINE STOPPED: Column-name formatting differences "
        "were detected for: %s",
        formatting_difference_columns
    )

    print("\nALERT: Human review is required.")

    print(
        "The pipeline cannot continue because one or more "
        "column names differ by capitalisation, spacing, "
        "punctuation or another formatting variation."
    )

    print(
        "\nAffected schema records:"
    )

    display(
        schema_comparison_df.loc[
            formatting_difference_mask
        ].reset_index(drop=True)
    )


# =============================================================================
# BLOCKING CONDITION 2 - UNKNOWN STATUSES
# =============================================================================

if unknown_status_columns:

    bronze_schema_logger.error(
        "PIPELINE STOPPED: Unrecognised schema statuses "
        "were detected for: %s",
        unknown_status_columns
    )

    print("\nALERT: Unknown schema status detected.")

    print(
        "The pipeline cannot continue because one or more "
        "schema results do not match an approved validation rule."
    )

    print(
        "\nAffected schema records:"
    )

    display(
        schema_comparison_df.loc[
            unknown_status_mask
        ].reset_index(drop=True)
    )


# =============================================================================
# NON-BLOCKING CONDITION - MISSING COLUMNS
# =============================================================================

if missing_columns:

    bronze_schema_logger.warning(
        "NON-BLOCKING SCHEMA ALERT: The following columns are "
        "missing from one or more child datasets: %s",
        missing_columns
    )

    print("\nALERT: Expected schema evolution detected.")

    print(
        "The following columns are not available in every "
        "child dataset:"
    )

    for column_name in missing_columns:
        print(f"  - {column_name}")

    print(
        "\nThe pipeline is allowed to continue because the remaining "
        "column names are consistent."
    )

    print(
        "During schema alignment, these missing columns will be "
        "added to the affected datasets with null values."
    )

    print(
        "\nAffected schema records:"
    )

    display(
        schema_comparison_df.loc[
            missing_column_mask
        ].reset_index(drop=True)
    )

else:

    bronze_schema_logger.info(
        "All child datasets contain the same column names."
    )


# =============================================================================
# FINAL BRONZE SCHEMA-GATE RESULT
# =============================================================================

bronze_schema_gate_passed = (
    not formatting_difference_columns
    and not unknown_status_columns
)

bronze_schema_gate_audit["gate_passed"] = (
    bronze_schema_gate_passed
)

bronze_schema_gate_audit["gate_status"] = (
    "PASS"
    if bronze_schema_gate_passed
    else "FAIL"
)


print("\n" + "-" * 100)

if bronze_schema_gate_passed:

    print("Bronze Schema Gate Result: PASS")

    print(
        "The child datasets may proceed to schema alignment "
        "and concatenation."
    )

else:

    print("Bronze Schema Gate Result: FAIL")

    print(
        "The child datasets must not be aligned or concatenated "
        "until the blocking schema issues have been reviewed."
    )

print("-" * 100)

2026-07-18 20:35:33,672 [WARNING] NON-BLOCKING SCHEMA ALERT: The following columns are missing from one or more child datasets: ['remaining_lease']



BRONZE SCHEMA VALIDATION GATE
Total columns reviewed            : 13
Exact-match columns               : 12
Columns missing from some sources : 1
Formatting differences detected   : 0
Unknown schema statuses           : 0

ALERT: Expected schema evolution detected.
The following columns are not available in every child dataset:
  - remaining_lease

The pipeline is allowed to continue because the remaining column names are consistent.
During schema alignment, these missing columns will be added to the affected datasets with null values.

Affected schema records:


,normalised_column_name,Child_1,Child_2,Child_3,Child_4,Child_5,schema_consistency
0,remaining_lease,remaining_lease,MISSING,MISSING,MISSING,remaining_lease,REVIEW - Missing from 3 dataset(s)



----------------------------------------------------------------------------------------------------
Bronze Schema Gate Result: PASS
The child datasets may proceed to schema alignment and concatenation.
----------------------------------------------------------------------------------------------------


### Bronze 5: Raw schema alignment

In [8]:
# =============================================================================
# Purpose:
# Align all extracted child datasets to one common schema before concatenation.
#
# Rules:
# - Continue only when the Bronze schema validation gate has passed.
# - Add approved missing columns as null values.
# - Reorder every child dataset to the same target schema.
# - Preserve the original raw Bronze DataFrames.
# - Record all alignment actions for auditing.
# =============================================================================

if not bronze_schema_gate_passed:
    raise RuntimeError(
        "Bronze schema alignment cannot begin because the "
        "Bronze schema validation gate failed."
    )


# =============================================================================
# DEFINE THE TARGET SCHEMA
# =============================================================================

# Use the normalised column names identified during schema comparison.
bronze_target_schema = (
    schema_comparison_df["normalised_column_name"]
    .tolist()
)

# Keep the technical lineage fields at the end.
for column in ["dataset_id", "vault_id"]:
    if column in bronze_target_schema:
        bronze_target_schema.remove(column)

bronze_target_schema.extend(["dataset_id", "vault_id"])

# Keep the technical lineage field at the end.
if "dataset_id" in bronze_target_schema:
    bronze_target_schema.remove("dataset_id")
    bronze_target_schema.append("dataset_id")


# =============================================================================
# ALIGN EACH CHILD DATASET
# =============================================================================

# Stores schema-aligned copies of the raw child datasets.
# The original DataFrames in bronze_child_datasets remain unchanged.
bronze_aligned_datasets: dict[str, pd.DataFrame] = {}

bronze_alignment_audit_records = []


for dataset_id, raw_bronze_df in bronze_child_datasets.items():

    # Create a copy so the original Bronze source data is preserved.
    aligned_bronze_df = raw_bronze_df.copy()

    # Add the actual source dataset identifier for lineage.
    aligned_bronze_df["dataset_id"] = dataset_id

    original_columns = list(
        aligned_bronze_df.columns
    )

    columns_added_as_null = []

    # Add any approved missing columns.
    for column_name in bronze_target_schema:

        if column_name not in aligned_bronze_df.columns:

            aligned_bronze_df[column_name] = pd.NA

            columns_added_as_null.append(
                column_name
            )

    # Reorder every child dataset to the same target schema.
    aligned_bronze_df = aligned_bronze_df[
        bronze_target_schema
    ]

    bronze_aligned_datasets[dataset_id] = (
        aligned_bronze_df
    )

    row_count_preserved = (
        len(raw_bronze_df)
        == len(aligned_bronze_df)
    )

    schema_matches_target = (
        list(aligned_bronze_df.columns)
        == bronze_target_schema
    )

    bronze_alignment_audit_records.append(
        {
            "dataset_id": dataset_id,
            "original_row_count": len(raw_bronze_df),
            "aligned_row_count": len(aligned_bronze_df),
            "original_column_count": len(original_columns),
            "aligned_column_count": len(
                aligned_bronze_df.columns
            ),
            "columns_added_as_null": columns_added_as_null,
            "row_count_preserved": row_count_preserved,
            "schema_matches_target": schema_matches_target
        }
    )

    if columns_added_as_null:

        bronze_schema_logger.warning(
            "Dataset %s: added missing columns as null values: %s",
            dataset_id,
            columns_added_as_null
        )

    else:

        bronze_schema_logger.info(
            "Dataset %s: no missing columns required.",
            dataset_id
        )


# =============================================================================
# CREATE ALIGNMENT AUDIT TABLE
# =============================================================================

bronze_alignment_audit_df = pd.DataFrame(
    bronze_alignment_audit_records
)


print("\n" + "=" * 100)
print("BRONZE SCHEMA ALIGNMENT AUDIT")
print("=" * 100)

display(bronze_alignment_audit_df)


# =============================================================================
# VALIDATE ALIGNMENT RESULTS
# =============================================================================

all_row_counts_preserved = (
    bronze_alignment_audit_df[
        "row_count_preserved"
    ].all()
)

all_schemas_match_target = (
    bronze_alignment_audit_df[
        "schema_matches_target"
    ].all()
)

bronze_alignment_passed = (
    all_row_counts_preserved
    and all_schemas_match_target
)


print("\n" + "-" * 100)

print(
    f"All row counts preserved : "
    f"{'PASS' if all_row_counts_preserved else 'FAIL'}"
)

print(
    f"All schemas aligned      : "
    f"{'PASS' if all_schemas_match_target else 'FAIL'}"
)

print(
    f"Bronze Alignment Result : "
    f"{'PASS' if bronze_alignment_passed else 'FAIL'}"
)

print("-" * 100)


if not bronze_alignment_passed:
    raise RuntimeError(
        "Bronze schema alignment failed. Review "
        "bronze_alignment_audit_df before continuing."
    )

2026-07-18 20:35:33,778 [INFO] Dataset d_8b84c4ee58e3cfc0ece0d773c8ca6abc: no missing columns required.
2026-07-18 20:35:33,881 [WARNING] Dataset d_43f493c6c50d54243cc1eab0df142d6a: added missing columns as null values: ['remaining_lease']
2026-07-18 20:35:33,894 [WARNING] Dataset d_2d5ff9ea31397b66239f245f57751537: added missing columns as null values: ['remaining_lease']
2026-07-18 20:35:33,966 [WARNING] Dataset d_ebc5ab87086db484f88045b47411ebc5: added missing columns as null values: ['remaining_lease']
2026-07-18 20:35:33,975 [INFO] Dataset d_ea9ed51da2787afaf8e51f827c304208: no missing columns required.



BRONZE SCHEMA ALIGNMENT AUDIT


,dataset_id,original_row_count,aligned_row_count,original_column_count,aligned_column_count,columns_added_as_null,row_count_preserved,schema_matches_target
0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,235808,235808,14,13,[],True,True
1,d_43f493c6c50d54243cc1eab0df142d6a,369651,369651,13,13,[remaining_lease],True,True
2,d_2d5ff9ea31397b66239f245f57751537,52203,52203,13,13,[remaining_lease],True,True
3,d_ebc5ab87086db484f88045b47411ebc5,287196,287196,13,13,[remaining_lease],True,True
4,d_ea9ed51da2787afaf8e51f827c304208,37153,37153,14,13,[],True,True



----------------------------------------------------------------------------------------------------
All row counts preserved : PASS
All schemas aligned      : PASS
Bronze Alignment Result : PASS
----------------------------------------------------------------------------------------------------


### Bronze 6: Create Bronze master dataset

In [9]:
# =============================================================================
# Purpose:
# Combine all schema-aligned Bronze child datasets into a single Bronze master
# dataset and verify that no records were lost during concatenation.
# =============================================================================

if not bronze_alignment_passed:
    raise RuntimeError(
        "Bronze master dataset creation cannot begin because "
        "schema alignment failed."
    )


# =============================================================================
# CREATE BRONZE MASTER DATASET
# =============================================================================

bronze_df = pd.concat(
    bronze_aligned_datasets.values(),
    ignore_index=True
)


# =============================================================================
# RECORD COUNT RECONCILIATION
# =============================================================================

expected_record_count = sum(
    len(child_df)
    for child_df in bronze_child_datasets.values()
)

actual_record_count = len(
    bronze_df
)


print("\n" + "=" * 100)
print("BRONZE MASTER DATASET RECONCILIATION")
print("=" * 100)

print(
    f"Expected Records "
    f"(Sum of Bronze Child Datasets) : "
    f"{expected_record_count:,}"
)

print(
    f"Actual Records "
    f"(Bronze Master Dataset)        : "
    f"{actual_record_count:,}"
)

print(
    f"Bronze Master Columns          : "
    f"{len(bronze_df.columns)}"
)


bronze_reconciliation_passed = (
    expected_record_count == actual_record_count
)

print(
    f"Record Count Validation        : "
    f"{'PASS' if bronze_reconciliation_passed else 'FAIL'}"
)

if not bronze_reconciliation_passed:

    raise RuntimeError(
        "Bronze record reconciliation failed. "
        "The Bronze master dataset row count does not equal "
        "the sum of all Bronze child datasets."
    )


print("\nBronze Master Dataset Preview\n")

display(
    bronze_df.head()
)


BRONZE MASTER DATASET RECONCILIATION
Expected Records (Sum of Bronze Child Datasets) : 982,011
Actual Records (Bronze Master Dataset)        : 982,011
Bronze Master Columns          : 13
Record Count Validation        : PASS

Bronze Master Dataset Preview



,block,flat_model,flat_type,floor_area_sqm,lease_commence_date,month,remaining_lease,resale_price,storey_range,street_name,town,vault_id,dataset_id
0,406,Improved,2 ROOM,44,1979,2017-01,61 years 04 months,232000,10 TO 12,ANG MO KIO AVE 10,ANG MO KIO,1,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
1,108,New Generation,3 ROOM,67,1978,2017-01,60 years 07 months,250000,01 TO 03,ANG MO KIO AVE 4,ANG MO KIO,2,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
2,602,New Generation,3 ROOM,67,1980,2017-01,62 years 05 months,262000,01 TO 03,ANG MO KIO AVE 5,ANG MO KIO,3,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
3,465,New Generation,3 ROOM,68,1980,2017-01,62 years 01 month,265000,04 TO 06,ANG MO KIO AVE 10,ANG MO KIO,4,d_8b84c4ee58e3cfc0ece0d773c8ca6abc
4,601,New Generation,3 ROOM,67,1980,2017-01,62 years 05 months,265000,01 TO 03,ANG MO KIO AVE 5,ANG MO KIO,5,d_8b84c4ee58e3cfc0ece0d773c8ca6abc


# Silver layer

### Silver 1: Create working silver dataset (isolated from Bronze)

In [10]:
silver_df = bronze_df.copy()

print("\n" + "=" * 100)
print("SILVER 1.1 - CREATE SILVER WORKING DATASET")
print("=" * 100)

print(
    f"Bronze rows : {len(bronze_df):,}"
)

print(
    f"Silver rows : {len(silver_df):,}"
)


SILVER 1.1 - CREATE SILVER WORKING DATASET
Bronze rows : 982,011
Silver rows : 982,011


### Silver 2: Data cleaning (standardise data types)

In [11]:
# =============================================================================
# STANDARDISE DATA TYPES
# =============================================================================

# Text columns
text_columns = [
    "dataset_id",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "flat_model",
    "remaining_lease"
]

for column in text_columns:

    if column in silver_df.columns:

        silver_df[column] = (
            silver_df[column]
            .astype("string")
            .str.strip()
        )


# Integer columns
integer_columns = [
    "vault_id",
    "lease_commence_date"
]

for column in integer_columns:

    if column in silver_df.columns:

        silver_df[column] = pd.to_numeric(
            silver_df[column],
            errors="coerce"
        ).astype("Int64")


# Decimal columns
decimal_columns = [
    "floor_area_sqm",
    "resale_price"
]

for column in decimal_columns:

    if column in silver_df.columns:

        silver_df[column] = pd.to_numeric(
            silver_df[column],
            errors="coerce"
        )


# Date columns
if "month" in silver_df.columns:

    silver_df["month"] = pd.to_datetime(
        silver_df["month"],
        format="%Y-%m",
        errors="coerce"
    )

# =============================================================================
# DISPLAY BRONZE VS SILVER DATA TYPES
# =============================================================================

bronze_dtype_df = (
    bronze_df.dtypes
    .rename("bronze_data_type")
    .reset_index()
    .rename(columns={"index": "column_name"})
)

silver_dtype_df = (
    silver_df.dtypes
    .rename("silver_data_type")
    .reset_index()
    .rename(columns={"index": "column_name"})
)

dtype_comparison_df = (
    bronze_dtype_df
    .merge(
        silver_dtype_df,
        on="column_name",
        how="inner"
    )
)

dtype_comparison_df["data_type_changed"] = (
    dtype_comparison_df["bronze_data_type"].astype(str)
    !=
    dtype_comparison_df["silver_data_type"].astype(str)
)

print("\nData type standardisation completed.\n")

display(dtype_comparison_df)


Data type standardisation completed.



,column_name,bronze_data_type,silver_data_type,data_type_changed
0,block,object,string[python],True
1,flat_model,object,string[python],True
2,flat_type,object,string[python],True
3,floor_area_sqm,object,float64,True
4,lease_commence_date,object,Int64,True
5,month,object,datetime64[ns],True
6,remaining_lease,object,string[python],True
7,resale_price,object,float64,True
8,storey_range,object,string[python],True
9,street_name,object,string[python],True


### Silver 3: Data cleaning (composite key resolution - keep highest resale price)

In [12]:
# =============================================================================
# Business Rule:
# Assume the composite key consists of all columns except resale_price.
# If duplicate keys exist, retain the record with the highest resale_price
# and move the lower-priced duplicate(s) to the failed dataset.
# =============================================================================

# Define the composite key
composite_key = [
    column
    for column in silver_df.columns
    if column != "resale_price"
]

# Sort so the highest resale price appears first
silver_sorted_df = (
    silver_df
    .sort_values(
        by="resale_price",
        ascending=False
    )
)

# Passed dataset
silver_passed_df = (
    silver_sorted_df
    .drop_duplicates(
        subset=composite_key,
        keep="first"
    )
    .reset_index(drop=True)
)

# Failed dataset
silver_failed_duplicate_df = (
    silver_sorted_df[
        silver_sorted_df.duplicated(
            subset=composite_key,
            keep="first"
        )
    ]
    .reset_index(drop=True)
)

# Summary
print(
    f"Original Records : {len(silver_df):,}"
)

print(
    f"Passed Records   : {len(silver_passed_df):,}"
)

print(
    f"Failed Records   : {len(silver_failed_duplicate_df):,}"
)

print(
    f"Duplicates Removed: "
    f"{len(silver_failed_duplicate_df):,}"
)

display(silver_failed_duplicate_df.head())

Original Records : 982,011
Passed Records   : 982,011
Failed Records   : 0
Duplicates Removed: 0


,block,flat_model,flat_type,floor_area_sqm,lease_commence_date,month,remaining_lease,resale_price,storey_range,street_name,town,vault_id,dataset_id


### Silver 4: Data cleaning (basic data standardisation)

In [13]:
# =============================================================================
# Purpose:
# 1. Standardise text fields for consistency.
# 2. Remove leading and trailing whitespace.
# 3. Replace multiple consecutive spaces with a single space.
# 4. Preserve missing values.
# 5. Preserve business meaning while improving data consistency.
# =============================================================================

silver_df = silver_df.copy()


# =============================================================================
# IDENTIFY TEXT COLUMNS
# =============================================================================

text_columns = silver_df.select_dtypes(
    include=[
        "object",
        "string"
    ]
).columns.tolist()


# =============================================================================
# CAPTURE PRE-CLEANING METRICS
# =============================================================================

pre_cleaning_null_counts = (
    silver_df[text_columns]
    .isna()
    .sum()
)

pre_cleaning_unique_counts = (
    silver_df[text_columns]
    .nunique(dropna=True)
)


# =============================================================================
# STANDARDISE GENERAL TEXT VALUES
# =============================================================================

for column in text_columns:

    silver_df[column] = (
        silver_df[column]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

    # Convert empty strings back to missing values.
    silver_df[column] = (
        silver_df[column]
        .replace(
            "",
            pd.NA
        )
    )


# =============================================================================
# STANDARDISE SELECTED CATEGORICAL FIELDS
# =============================================================================

uppercase_columns = [
    "town",
    "flat_type",
    "flat_model",
    "storey_range"
]

for column in uppercase_columns:

    if column in silver_df.columns:

        silver_df[column] = (
            silver_df[column]
            .str.upper()
        )


# =============================================================================
# STANDARDISE STREET NAMES
# =============================================================================

if "street_name" in silver_df.columns:

    silver_df["street_name"] = (
        silver_df["street_name"]
        .str.title()
    )


# =============================================================================
# CAPTURE POST-CLEANING METRICS
# =============================================================================

post_cleaning_null_counts = (
    silver_df[text_columns]
    .isna()
    .sum()
)

post_cleaning_unique_counts = (
    silver_df[text_columns]
    .nunique(dropna=True)
)


# =============================================================================
# CREATE CLEANING AUDIT SUMMARY
# =============================================================================

silver_basic_cleaning_audit_df = pd.DataFrame(
    {
        "column": text_columns,
        "null_count_before": [
            pre_cleaning_null_counts[column]
            for column in text_columns
        ],
        "null_count_after": [
            post_cleaning_null_counts[column]
            for column in text_columns
        ],
        "unique_values_before": [
            pre_cleaning_unique_counts[column]
            for column in text_columns
        ],
        "unique_values_after": [
            post_cleaning_unique_counts[column]
            for column in text_columns
        ]
    }
)

silver_basic_cleaning_audit_df[
    "unique_values_reduced"
] = (
    silver_basic_cleaning_audit_df[
        "unique_values_before"
    ]
    - silver_basic_cleaning_audit_df[
        "unique_values_after"
    ]
)


# =============================================================================
# DISPLAY CLEANED SAMPLE
# =============================================================================

preview_columns = [
    column
    for column in [
        "town",
        "flat_type",
        "block",
        "street_name",
        "storey_range",
        "flat_model"
    ]
    if column in silver_df.columns
]

print("=" * 100)
print("SILVER BASIC DATA STANDARDISATION")
print("=" * 100)

print(
    f"Text columns standardised : "
    f"{len(text_columns)}"
)

print(
    "Missing values preserved : "
    f"{pre_cleaning_null_counts.sum() == post_cleaning_null_counts.sum()}"
)

print("\nCleaned data preview:")

display(
    silver_df[
        preview_columns
    ].head()
)

print("\nCleaning audit summary:")

display(
    silver_basic_cleaning_audit_df
)

SILVER BASIC DATA STANDARDISATION
Text columns standardised : 8
Missing values preserved : True

Cleaned data preview:


,town,flat_type,block,street_name,storey_range,flat_model
0,ANG MO KIO,2 ROOM,406,Ang Mo Kio Ave 10,10 TO 12,IMPROVED
1,ANG MO KIO,3 ROOM,108,Ang Mo Kio Ave 4,01 TO 03,NEW GENERATION
2,ANG MO KIO,3 ROOM,602,Ang Mo Kio Ave 5,01 TO 03,NEW GENERATION
3,ANG MO KIO,3 ROOM,465,Ang Mo Kio Ave 10,04 TO 06,NEW GENERATION
4,ANG MO KIO,3 ROOM,601,Ang Mo Kio Ave 5,01 TO 03,NEW GENERATION



Cleaning audit summary:


,column,null_count_before,null_count_after,unique_values_before,unique_values_after,unique_values_reduced
0,block,0,0,2772,2772,0
1,flat_model,0,0,34,21,13
2,flat_type,0,0,8,8,0
3,remaining_lease,709050,709050,750,750,0
4,storey_range,0,0,25,25,0
5,street_name,0,0,595,595,0
6,town,0,0,27,27,0
7,dataset_id,0,0,5,5,0


### Silver 5: Data profiling (completeness, uniqueness, and potential data quality issues)

In [14]:
# Profile every column to identify completeness, uniqueness and potential data-quality issues.

profile_records = []

total_rows = len(silver_df)

for column in silver_df.columns:

    series = silver_df[column]

    null_count = series.isna().sum()

    profile_records.append(
        {
            "column_name": column,
            "data_type": str(series.dtype),
            "row_count": total_rows,
            "null_count": null_count,
            "null_percentage": round(
                (null_count / total_rows) * 100,
                2
            ),
            "distinct_values": series.nunique(
                dropna=True
            ),
            "duplicate_values": (
                total_rows
                - series.nunique(dropna=False)
            ),
            "minimum": (
                series.min()
                if pd.api.types.is_numeric_dtype(series)
                else None
            ),
            "maximum": (
                series.max()
                if pd.api.types.is_numeric_dtype(series)
                else None
            )
        }
    )

silver_column_profile_df = pd.DataFrame(
    profile_records
)

display(silver_column_profile_df)

,column_name,data_type,row_count,null_count,null_percentage,distinct_values,duplicate_values,minimum,maximum
0,block,string,982011,0,0.0,2772,979239,NaN,NaN
1,flat_model,string,982011,0,0.0,21,981990,NaN,NaN
2,flat_type,string,982011,0,0.0,8,982003,NaN,NaN
3,floor_area_sqm,float64,982011,0,0.0,232,981779,28.0,366.7
4,lease_commence_date,Int64,982011,0,0.0,57,981954,1966.0,2022.0
5,month,datetime64[ns],982011,0,0.0,439,981572,NaN,NaN
6,remaining_lease,string,982011,709050,72.2,750,981260,NaN,NaN
7,resale_price,float64,982011,0,0.0,10274,971737,5000.0,1728000.0
8,storey_range,string,982011,0,0.0,25,981986,NaN,NaN
9,street_name,string,982011,0,0.0,595,981416,NaN,NaN


### Silver 6: Data profiling (source record integrity)

In [15]:
# Verify that the source record identifiers remain unique within each
# source dataset after Bronze processing so vault_id and dataset_id in combination 
# can be used as a composite key to uniquely identify each record.

source_record_integrity_df = (
    bronze_df
    .groupby("dataset_id")
    .agg(
        total_records=("vault_id", "count"),
        unique_vault_ids=("vault_id", "nunique")
    )
)

source_record_integrity_df["vault_id_unique"] = (
    source_record_integrity_df["total_records"]
    == source_record_integrity_df["unique_vault_ids"]
)

display(source_record_integrity_df)

if source_record_integrity_df["vault_id_unique"].all():
    print("\nPASS: vault_id is unique within every source dataset.")
else:
    print("\nWARNING: Duplicate vault_id values detected within one or more source datasets.")

,total_records,unique_vault_ids,vault_id_unique
dataset_id,,,
d_2d5ff9ea31397b66239f245f57751537,52203,52203,True
d_43f493c6c50d54243cc1eab0df142d6a,369651,369651,True
d_8b84c4ee58e3cfc0ece0d773c8ca6abc,235808,235808,True
d_ea9ed51da2787afaf8e51f827c304208,37153,37153,True
d_ebc5ab87086db484f88045b47411ebc5,287196,287196,True



PASS: vault_id is unique within every source dataset.


### Silver 7: Data quality (Great Expectations setup)

In [16]:
import great_expectations as gx

print(gx.__version__)

2026-07-18 20:35:46,652 [INFO] Skipping registering function get_context because it does not have a class
2026-07-18 20:35:47,746 [INFO] Skipping registering function DataSourceManager._register_add_datasource.<locals>.crud_method_info because it is a closure
2026-07-18 20:35:47,747 [INFO] Skipping registering function DataSourceManager._register_update_datasource.<locals>.crud_method_info because it is a closure
2026-07-18 20:35:47,747 [INFO] Skipping registering function DataSourceManager._register_add_or_update_datasource.<locals>.crud_method_info because it is a closure
2026-07-18 20:35:47,747 [INFO] Skipping registering function DataSourceManager._register_delete_datasource.<locals>.crud_method_info because it is a closure
2026-07-18 20:35:47,749 [INFO] Skipping registering function DataSourceManager._register_add_datasource.<locals>.crud_method_info because it is a closure
2026-07-18 20:35:47,749 [INFO] Skipping registering function DataSourceManager._register_update_datasource.<

1.19.0


In [17]:
# =============================================================================
# SILVER 2.1 - GREAT EXPECTATIONS DATA QUALITY VALIDATION
# =============================================================================

import great_expectations as gx


# =============================================================================
# CREATE GX CONTEXT
# =============================================================================

gx_context = gx.get_context()


# =============================================================================
# CREATE PANDAS DATA SOURCE
# =============================================================================

gx_data_source = gx_context.data_sources.add_pandas(
    name="silver_pandas_source"
)


# =============================================================================
# CREATE DATA ASSET
# =============================================================================

gx_data_asset = gx_data_source.add_dataframe_asset(
    name="hdb_resale_silver_asset"
)


# =============================================================================
# CREATE BATCH DEFINITION
# =============================================================================

gx_batch_definition = (
    gx_data_asset.add_batch_definition_whole_dataframe(
        name="silver_whole_dataframe"
    )
)


# =============================================================================
# CREATE BATCH
# =============================================================================

gx_batch = gx_batch_definition.get_batch(
    batch_parameters={
        "dataframe": silver_df
    }
)

print("Great Expectations Batch created successfully.")

2026-07-18 20:35:48,089 [INFO] Could not find local file-backed GX project
2026-07-18 20:35:48,090 [INFO] Created temporary directory '/var/folders/pl/l5m8srb54vb0hqlcp588dpr00000gn/T/tmp8qu4n_xa' for ephemeral docs site
2026-07-18 20:35:48,090 [INFO] Loading 'datasources' ->
[]


Great Expectations Batch created successfully.


### Silver 8: Data quality (validation rules)

In [18]:
silver_expectations = []

validation_rules = {

    "town": sorted(
        silver_df["town"]
        .dropna()
        .unique()
        .tolist()
    ),

    "flat_type": sorted(
        silver_df["flat_type"]
        .dropna()
        .unique()
        .tolist()
    ),

    "flat_model": sorted(
        silver_df["flat_model"]
        .dropna()
        .unique()
        .tolist()
    ),

    "storey_range": sorted(
        silver_df["storey_range"]
        .dropna()
        .unique()
        .tolist()
    ),

    "month": {
        "minimum": silver_df["month"].min(),
        "maximum": silver_df["month"].max()
    }

}


print("=" * 100)
print("DATA VALIDATION RULES")
print("=" * 100)

for column_name, rule in validation_rules.items():

    print(f"\n{column_name}")

    if isinstance(rule, dict):

        print(rule)

    else:

        print(rule)

# =============================================================================
# EXPECTATION GROUP 1 - MANDATORY VALUES (no nulls)
# =============================================================================

mandatory_columns = [
    "dataset_id",
    "vault_id",
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "resale_price"
]

for column in mandatory_columns:

    silver_expectations.append(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=column
        )
    )


# =============================================================================
# EXPECTATION GROUP 2 - SOURCE RECORD INTEGRITY
# =============================================================================

silver_expectations.append(
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=[
            "dataset_id",
            "vault_id"
        ]
    )
)


# =============================================================================
# EXPECTATION GROUP 3 - NUMERIC RANGES
# =============================================================================

silver_expectations.extend([
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="floor_area_sqm",
        min_value=1
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="resale_price",
        min_value=1
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="lease_commence_date",
        min_value=1960,
        max_value=2026
    )
])


# =============================================================================
# EXPECTATION GROUP 4 - CONTROLLED VOCABULARIES
# =============================================================================

for column in [
    "town",
    "flat_type",
    "flat_model",
    "storey_range"
]:

    silver_expectations.append(
        gx.expectations.ExpectColumnValuesToBeInSet(
            column=column,
            value_set=validation_rules[column]
        )
    )


# Date range
silver_expectations.append(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="month",
        min_value=validation_rules["month"]["minimum"],
        max_value=validation_rules["month"]["maximum"]
    )
)

print(
    f"{len(silver_expectations)} "
    "Great Expectations rules configured."
)

DATA VALIDATION RULES

town
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'LIM CHU KANG', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']

flat_type
['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI GENERATION', 'MULTI-GENERATION']

flat_model
['2-ROOM', '3GEN', 'ADJOINED FLAT', 'APARTMENT', 'DBSS', 'IMPROVED', 'IMPROVED-MAISONETTE', 'MAISONETTE', 'MODEL A', 'MODEL A-MAISONETTE', 'MODEL A2', 'MULTI GENERATION', 'NEW GENERATION', 'PREMIUM APARTMENT', 'PREMIUM APARTMENT LOFT', 'PREMIUM MAISONETTE', 'SIMPLIFIED', 'STANDARD', 'TERRACE', 'TYPE S1', 'TYPE S2']

storey_range
['01 TO 03', '01 TO 05', '04 TO 06', '06 TO 10', '07 TO 09', '10 TO 12', '11 TO 15', '13 TO 15', '16 TO 18', '16 TO 20', '19 TO 21', '21 TO 25', 

### Silver 9: Data quality (validation)

In [19]:
silver_validation_records = []

for expectation in silver_expectations:

    result = gx_batch.validate(
        expectation
    )

    expectation_config = (
        result.expectation_config
    )

    silver_validation_records.append(
        {
            "expectation_type": (
                expectation_config.type
            ),
            "column": (
                expectation_config.kwargs.get(
                    "column"
                )
            ),
            "column_list": (
                expectation_config.kwargs.get(
                    "column_list"
                )
            ),
            "success": result.success,
            "element_count": (
                result.result.get(
                    "element_count"
                )
            ),
            "unexpected_count": (
                result.result.get(
                    "unexpected_count"
                )
            ),
            "unexpected_percent": (
                result.result.get(
                    "unexpected_percent"
                )
            )
        }
    )


silver_validation_summary_df = pd.DataFrame(
    silver_validation_records
)

display(silver_validation_summary_df)


silver_validation_passed = (
    silver_validation_summary_df[
        "success"
    ].all()
)

print(
    "\nSilver Data Quality Result: "
    f"{'PASS' if silver_validation_passed else 'FAIL'}"
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,expectation_type,column,column_list,success,element_count,unexpected_count,unexpected_percent
0,expect_column_values_to_not_be_null,dataset_id,None,True,982011,0,0.0
1,expect_column_values_to_not_be_null,vault_id,None,True,982011,0,0.0
2,expect_column_values_to_not_be_null,month,None,True,982011,0,0.0
3,expect_column_values_to_not_be_null,town,None,True,982011,0,0.0
4,expect_column_values_to_not_be_null,flat_type,None,True,982011,0,0.0
5,expect_column_values_to_not_be_null,block,None,True,982011,0,0.0
6,expect_column_values_to_not_be_null,street_name,None,True,982011,0,0.0
7,expect_column_values_to_not_be_null,storey_range,None,True,982011,0,0.0
8,expect_column_values_to_not_be_null,floor_area_sqm,None,True,982011,0,0.0
9,expect_column_values_to_not_be_null,flat_model,None,True,982011,0,0.0



Silver Data Quality Result: PASS


### Silver 10: Data quality (quality gate)

In [21]:
print("=" * 100)
print("SILVER DATA QUALITY GATE")
print("=" * 100)

if silver_validation_passed:

    silver_cleaned_df = silver_passed_df.copy()

    print("Quality Gate Result : PASS")
    print("All Great Expectations validation rules passed.")
    print("Dataset promoted to Silver Cleaned layer.")

else:

    print("Quality Gate Result : FAIL")
    print("One or more data quality validation rules failed.")

    failed_expectations_df = (
        silver_validation_summary_df[
            silver_validation_summary_df["success"] == False
        ]
    )

    display(failed_expectations_df)

    raise RuntimeError(
        "Silver Quality Gate failed. "
        "Dataset cannot proceed to business transformations."
    )

SILVER DATA QUALITY GATE
Quality Gate Result : PASS
All Great Expectations validation rules passed.
Dataset promoted to Silver Cleaned layer.


### Silver 11: Data transformation (recompute remaining_lease)

In [22]:
silver_transformed_df = silver_cleaned_df.copy()

# Preserve the source-provided remaining lease column
if (
    "remaining_lease" in silver_transformed_df.columns
    and
    "precomputed_remaining_lease"
    not in silver_transformed_df.columns
):

    silver_transformed_df = silver_transformed_df.rename(
        columns={
            "remaining_lease":
            "precomputed_remaining_lease"
        }
    )


def calculate_remaining_lease(
    lease_commence_date
):
    """
    Recompute the remaining lease based on a 99-year lease.
    The result is rounded down to complete years and months.
    """

    if pd.isna(lease_commence_date):
        return pd.NA

    lease_start_date = date(
        int(lease_commence_date),
        1,
        1
    )

    lease_end_date = lease_start_date.replace(
        year=lease_start_date.year + 99
    )

    today = date.today()

    if lease_end_date <= today:
        return "0 years 0 months"

    remaining = relativedelta(
        lease_end_date,
        today
    )

    return (
        f"{remaining.years} years "
        f"{remaining.months} months"
    )


silver_transformed_df["remaining_lease"] = (
    silver_transformed_df[
        "lease_commence_date"
    ].apply(calculate_remaining_lease)
)

print("Remaining lease recomputation completed.")

display(
    silver_transformed_df[
        [
            "lease_commence_date",
            "precomputed_remaining_lease",
            "remaining_lease"
        ]
    ].head()
)

Remaining lease recomputation completed.


,lease_commence_date,precomputed_remaining_lease,remaining_lease
0,2019,92 years 01 month,91 years 5 months
1,2016,89 years 03 months,88 years 5 months
2,2016,89 years 11 months,88 years 5 months
3,2016,88 years 11 months,88 years 5 months
4,2016,88 years 10 months,88 years 5 months


### Silver 12: Data transformation (add new column resale identifier)

In [24]:
# =============================================================================
# Format:
# S + Block(3 digits) + AvgPrice(2 digits) + Month(2 digits) + Town(1 character)
#
# Example:
# S0192301A
#
# Input:
# silver_transformed_df (from Silver 10)
# =============================================================================

# -------------------------------------------------------
# Block Component
# -------------------------------------------------------

silver_transformed_df["block_digits"] = (
    silver_transformed_df["block"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str[:3]
    .str.zfill(3)
)

# -------------------------------------------------------
# Average Resale Price Component
# -------------------------------------------------------

average_price = (
    silver_transformed_df
    .groupby(
        [
            "month",
            "town",
            "flat_type"
        ]
    )["resale_price"]
    .transform("mean")
)

silver_transformed_df["average_price_digits"] = (
    average_price
    .round(0)
    .astype(int)
    .astype(str)
    .str[:2]
    .str.zfill(2)
)

# -------------------------------------------------------
# Month Component
# -------------------------------------------------------

silver_transformed_df["month_digits"] = (
    pd.to_datetime(
        silver_transformed_df["month"]
    )
    .dt.strftime("%m")
)

# -------------------------------------------------------
# Town Component
# -------------------------------------------------------

silver_transformed_df["town_character"] = (
    silver_transformed_df["town"]
    .str[0]
    .str.upper()
)

# -------------------------------------------------------
# Final Resale Identifier
# -------------------------------------------------------

silver_transformed_df["resale_identifier"] = (
    "S"
    + silver_transformed_df["block_digits"]
    + silver_transformed_df["average_price_digits"]
    + silver_transformed_df["month_digits"]
    + silver_transformed_df["town_character"]
)

# -------------------------------------------------------
# Remove Helper Columns
# -------------------------------------------------------

silver_transformed_df.drop(
    columns=[
        "block_digits",
        "average_price_digits",
        "month_digits",
        "town_character"
    ],
    inplace=True
)

print("Resale Identifier successfully generated.")

print(
    f"Transformed records: "
    f"{len(silver_transformed_df):,}"
)

display(
    silver_transformed_df[
        [
            "block",
            "town",
            "flat_type",
            "month",
            "resale_price",
            "remaining_lease",
            "resale_identifier"
        ]
    ].head()
)

Resale Identifier successfully generated.
Transformed records: 982,011


,block,town,flat_type,month,resale_price,remaining_lease,resale_identifier
0,96A,BUKIT MERAH,5 ROOM,2026-04-01,1728000.0,91 years 5 months,S0961004B
1,92,QUEENSTOWN,5 ROOM,2026-02-01,1700000.0,88 years 5 months,S0921102Q
2,92,QUEENSTOWN,5 ROOM,2025-06-01,1658888.0,88 years 5 months,S0921206Q
3,93,QUEENSTOWN,5 ROOM,2026-06-01,1650000.0,88 years 5 months,S0931206Q
4,9A,BUKIT MERAH,5 ROOM,2026-03-01,1648888.0,88 years 5 months,S0091003B


### Silver 13: Data transformation (hash resale_identifier with clean data)

In [35]:
# =============================================================================
# Purpose:
# 1. Hash the resale identifier using SHA-256.
# 2. Confirm that record counts are preserved.
# 3. Confirm that duplicate source identifiers remain duplicate after hashing.
# 4. Confirm that no hash collisions were introduced.
#
# SHA-256 produces a deterministic, irreversible, fixed-length
# 64-character hexadecimal hash.
# =============================================================================

import hashlib

import pandas as pd
from IPython.display import display


# =============================================================================
# VALIDATE REQUIRED INPUT
# =============================================================================

required_columns = {
    "resale_identifier"
}

missing_required_columns = (
    required_columns
    - set(silver_transformed_df.columns)
)

if missing_required_columns:
    raise KeyError(
        "silver_transformed_df is missing the required columns: "
        f"{sorted(missing_required_columns)}"
    )


# =============================================================================
# SHA-256 HASH FUNCTION
# =============================================================================

def generate_sha256(value: object) -> str | pd._libs.missing.NAType:
    """
    Return the SHA-256 hexadecimal hash of a non-null value.

    Missing values remain missing and are not converted into text.
    """

    if pd.isna(value):
        return pd.NA

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


# =============================================================================
# CREATE HASHED WORKING DATASET
# =============================================================================

silver_hashed_df = silver_transformed_df.copy()

silver_hashed_df["hashed_resale_identifier"] = (
    silver_hashed_df["resale_identifier"]
    .apply(generate_sha256)
    .astype("string")
)


# =============================================================================
# HASH VALIDATION METRICS
# =============================================================================

source_record_count = len(silver_hashed_df)

hashed_record_count = (
    silver_hashed_df[
        "hashed_resale_identifier"
    ]
    .notna()
    .sum()
)

missing_source_identifier_count = (
    silver_hashed_df[
        "resale_identifier"
    ]
    .isna()
    .sum()
)

missing_hashed_identifier_count = (
    silver_hashed_df[
        "hashed_resale_identifier"
    ]
    .isna()
    .sum()
)

source_unique_count = (
    silver_hashed_df[
        "resale_identifier"
    ]
    .nunique(dropna=True)
)

hashed_unique_count = (
    silver_hashed_df[
        "hashed_resale_identifier"
    ]
    .nunique(dropna=True)
)

duplicate_source_identifier_count = (
    silver_hashed_df[
        "resale_identifier"
    ]
    .notna()
    .sum()
    - source_unique_count
)

duplicate_hashed_identifier_count = (
    silver_hashed_df[
        "hashed_resale_identifier"
    ]
    .notna()
    .sum()
    - hashed_unique_count
)

hash_collision_count = max(
    source_unique_count
    - hashed_unique_count,
    0
)


# =============================================================================
# VALIDATE HASH FORMAT
# =============================================================================

valid_hash_format_mask = (
    silver_hashed_df[
        "hashed_resale_identifier"
    ]
    .dropna()
    .str.fullmatch(
        r"[0-9a-f]{64}"
    )
)

invalid_hash_format_count = (
    (~valid_hash_format_mask)
    .sum()
)


# =============================================================================
# VALIDATE DETERMINISTIC HASHING
# =============================================================================
# Each source identifier must map to exactly one hashed identifier.

identifier_hash_mapping_df = (
    silver_hashed_df
    .dropna(
        subset=[
            "resale_identifier",
            "hashed_resale_identifier"
        ]
    )
    .groupby(
        "resale_identifier"
    )["hashed_resale_identifier"]
    .nunique()
    .reset_index(
        name="hashed_values_per_identifier"
    )
)

non_deterministic_identifier_count = (
    identifier_hash_mapping_df[
        "hashed_values_per_identifier"
    ]
    .gt(1)
    .sum()
)


# =============================================================================
# OVERALL HASH VALIDATION RESULT
# =============================================================================

record_count_preserved = (
    len(silver_hashed_df)
    == len(silver_transformed_df)
)

null_count_preserved = (
    missing_source_identifier_count
    == missing_hashed_identifier_count
)

duplicate_pattern_preserved = (
    duplicate_source_identifier_count
    == duplicate_hashed_identifier_count
)

hash_validation_passed = all(
    [
        record_count_preserved,
        null_count_preserved,
        duplicate_pattern_preserved,
        hash_collision_count == 0,
        invalid_hash_format_count == 0,
        non_deterministic_identifier_count == 0
    ]
)


# =============================================================================
# CREATE HASH VALIDATION AUDIT TABLE
# =============================================================================

silver_hash_validation_audit_df = pd.DataFrame(
    [
        {
            "total_records": source_record_count,
            "non_null_hashed_records": hashed_record_count,
            "missing_source_identifiers": (
                missing_source_identifier_count
            ),
            "missing_hashed_identifiers": (
                missing_hashed_identifier_count
            ),
            "unique_source_identifiers": (
                source_unique_count
            ),
            "duplicate_source_identifiers": (
                duplicate_source_identifier_count
            ),
            "unique_hashed_identifiers": (
                hashed_unique_count
            ),
            "duplicate_hashed_identifiers": (
                duplicate_hashed_identifier_count
            ),
            "hash_collisions": hash_collision_count,
            "invalid_hash_formats": (
                invalid_hash_format_count
            ),
            "non_deterministic_identifiers": (
                non_deterministic_identifier_count
            ),
            "validation_status": (
                "PASS"
                if hash_validation_passed
                else "FAIL"
            )
        }
    ]
)


# =============================================================================
# PRINT VALIDATION SUMMARY
# =============================================================================

print("\nSHA-256 Hash Validation")
print("=" * 80)

print(
    f"Total Records                 : "
    f"{source_record_count:,}"
)

print(
    f"Unique Resale Identifiers     : "
    f"{source_unique_count:,}"
)

print(
    f"Duplicate Resale Identifiers  : "
    f"{duplicate_source_identifier_count:,}"
)

print(
    f"Unique Hashed Identifiers     : "
    f"{hashed_unique_count:,}"
)

print(
    f"Duplicate Hashed Identifiers  : "
    f"{duplicate_hashed_identifier_count:,}"
)

print(
    f"Missing Source Identifiers    : "
    f"{missing_source_identifier_count:,}"
)

print(
    f"Missing Hashed Identifiers    : "
    f"{missing_hashed_identifier_count:,}"
)

print(
    f"Hash Collisions               : "
    f"{hash_collision_count:,}"
)

print(
    f"Invalid SHA-256 Formats       : "
    f"{invalid_hash_format_count:,}"
)

print(
    f"Non-Deterministic Mappings    : "
    f"{non_deterministic_identifier_count:,}"
)

print(
    "\nHash Validation Result        : "
    f"{'PASS' if hash_validation_passed else 'FAIL'}"
)


# =============================================================================
# STOP PIPELINE IF HASH VALIDATION FAILS
# =============================================================================

if not hash_validation_passed:

    display(
        silver_hash_validation_audit_df
    )

    raise RuntimeError(
        "SHA-256 hash validation failed. "
        "The hashed dataset cannot be promoted to the output layer."
    )


# =============================================================================
# CREATE FINAL HASHED OUTPUT DATASET
# =============================================================================
# The plaintext resale identifier is excluded from the protected output.

silver_hashed_output_df = (
    silver_hashed_df
    .drop(
        columns=[
            "resale_identifier"
        ]
    )
    .copy()
)


# =============================================================================
# SAMPLE OUTPUT
# =============================================================================

sample_columns = [
    column
    for column in [
        "block",
        "town",
        "flat_type",
        "month",
        "resale_identifier",
        "hashed_resale_identifier"
    ]
    if column in silver_hashed_df.columns
]

print("\nSample Hashed Records")
print("=" * 80)

display(
    silver_hashed_df[
        sample_columns
    ].head()
)

print("\nHash Validation Audit")

display(
    silver_hash_validation_audit_df
)


SHA-256 Hash Validation
Total Records                 : 982,011
Unique Resale Identifiers     : 650,116
Duplicate Resale Identifiers  : 331,895
Unique Hashed Identifiers     : 650,116
Duplicate Hashed Identifiers  : 331,895
Missing Source Identifiers    : 0
Missing Hashed Identifiers    : 0
Hash Collisions               : 0
Invalid SHA-256 Formats       : 0
Non-Deterministic Mappings    : 0

Hash Validation Result        : PASS

Sample Hashed Records


,block,town,flat_type,month,resale_identifier,hashed_resale_identifier
0,96A,BUKIT MERAH,5 ROOM,2026-04-01,S0961004B,acf02627fe56dc5b7fb5dd8656b2165840840e43fcfa6d...
1,92,QUEENSTOWN,5 ROOM,2026-02-01,S0921102Q,fe8c51e42b8e500e229d8b880a01c0db4aff1fe768a8fe...
2,92,QUEENSTOWN,5 ROOM,2025-06-01,S0921206Q,b402eb5dc0d00f534459113590b8eb519ad7aba78a6d64...
3,93,QUEENSTOWN,5 ROOM,2026-06-01,S0931206Q,99081ff00e6ce3208acfb84e1880f78e213ca6c0430813...
4,9A,BUKIT MERAH,5 ROOM,2026-03-01,S0091003B,bd82d896c8e514d3312fa1c9d46c1d1bfe7a2bd640125b...



Hash Validation Audit


,total_records,non_null_hashed_records,missing_source_identifiers,missing_hashed_identifiers,unique_source_identifiers,duplicate_source_identifiers,unique_hashed_identifiers,duplicate_hashed_identifiers,hash_collisions,invalid_hash_formats,non_deterministic_identifiers,validation_status
0,982011,982011,0,0,650116,331895,650116,331895,0,0,0,PASS


### Silver 14: Data protection techniques (comparison)

In [26]:
# =============================================================================

sample_identifier = "S0192301A"

data_protection_comparison_df = pd.DataFrame(
    {
        "technique": [
            "Original",
            "Data Masking",
            "Pseudo-anonymisation",
            "SHA-256 Hashing"
        ],
        "example_output": [
            sample_identifier,
            "S0******A",
            "RID000001",
            generate_sha256(sample_identifier)
        ],
        "brief_description": [
            "Original readable identifier.",
            "Conceals part of the identifier while retaining its general format.",
            "Replaces the identifier with an artificial value linked through a mapping table.",
            "Transforms the identifier into a fixed-length irreversible hash."
        ],
        "reversible": [
            "N/A",
            "Partially",
            "Yes, with mapping table",
            "No"
        ],
        "record_linkage": [
            "Yes",
            "Usually no",
            "Yes",
            "Yes"
        ],
        "uniqueness_preserved": [
            "Yes",
            "Not guaranteed",
            "Yes, if pseudonyms are unique",
            "Yes, subject to collision validation"
        ]
    }
)

print("\nData Protection Technique Comparison")
print("=" * 80)

display(data_protection_comparison_df)

print(
    "\nConclusion:\n"
    "- Data masking conceals values while retaining part of the original format, "
    "but uniqueness and reliable record linkage may be lost.\n"
    "- Pseudo-anonymisation replaces identifiers with artificial values and can "
    "preserve uniqueness and linkage, but remains reversible through a mapping table.\n"
    "- SHA-256 hashing produces irreversible and deterministic values. The same "
    "identifier produces the same hash, enabling record linkage, while collision "
    "validation confirms that uniqueness has been preserved."
)


Data Protection Technique Comparison


,technique,example_output,brief_description,reversible,record_linkage,uniqueness_preserved
0,Original,S0192301A,Original readable identifier.,N/A,Yes,Yes
1,Data Masking,S0******A,Conceals part of the identifier while retainin...,Partially,Usually no,Not guaranteed
2,Pseudo-anonymisation,RID000001,Replaces the identifier with an artificial val...,"Yes, with mapping table",Yes,"Yes, if pseudonyms are unique"
3,SHA-256 Hashing,448f39f057bbe98860987d8d529913af05548f08fb3395...,Transforms the identifier into a fixed-length ...,No,Yes,"Yes, subject to collision validation"



Conclusion:
- Data masking conceals values while retaining part of the original format, but uniqueness and reliable record linkage may be lost.
- Pseudo-anonymisation replaces identifiers with artificial values and can preserve uniqueness and linkage, but remains reversible through a mapping table.
- SHA-256 hashing produces irreversible and deterministic values. The same identifier produces the same hash, enabling record linkage, while collision validation confirms that uniqueness has been preserved.


# Gold layer

### Gold 1: Strategic Planning Team - anomaly detection (resale price)

In [27]:
# =============================================================================
# Heuristic:
# Potential resale price anomalies are identified using the IQR method.
#
# Lower Bound = Q1 - (1.5 × IQR)
# Upper Bound = Q3 + (1.5 × IQR)
#
# If the calculated lower bound is below the minimum resale price in the
# dataset, the minimum resale price is used as the practical lower threshold.
# =============================================================================

import numpy as np

# Calculate quartiles
q1 = silver_df["resale_price"].quantile(0.25)
q3 = silver_df["resale_price"].quantile(0.75)

# Calculate Interquartile Range (IQR)
iqr = q3 - q1

# Calculate IQR thresholds
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

# Practical lower threshold
display_lower_bound = max(
    silver_df["resale_price"].min(),
    lower_bound
)

# Identify anomalies
silver_anomaly_df = silver_df[
    (
        silver_df["resale_price"] < display_lower_bound
    )
    |
    (
        silver_df["resale_price"] > upper_bound
    )
].copy()

# Identify normal records
silver_non_anomaly_df = silver_df[
    (
        silver_df["resale_price"] >= display_lower_bound
    )
    &
    (
        silver_df["resale_price"] <= upper_bound
    )
].copy()

# Prepare anomaly report
silver_anomaly_display_df = silver_anomaly_df[
    [
        "town",
        "flat_type",
        "month",
        "floor_area_sqm",
        "lease_commence_date",
        "resale_price"
    ]
].copy()

silver_anomaly_display_df["lower_threshold"] = display_lower_bound
silver_anomaly_display_df["upper_threshold"] = upper_bound

silver_anomaly_display_df["anomaly_type"] = np.where(
    silver_anomaly_display_df["resale_price"] > upper_bound,
    "Above Upper Threshold",
    "Below Lower Threshold"
)

silver_anomaly_display_df["threshold_difference"] = np.where(
    silver_anomaly_display_df["resale_price"] > upper_bound,
    silver_anomaly_display_df["resale_price"] - upper_bound,
    display_lower_bound - silver_anomaly_display_df["resale_price"]
).round(2)

# Sort by largest deviation
silver_anomaly_display_df = (
    silver_anomaly_display_df
    .sort_values(
        by="threshold_difference",
        ascending=False
    )
    .reset_index(drop=True)
)

# Summary
print("\nPotential Resale Price Anomalies")
print("=" * 80)

print(f"Lower Threshold       : ${display_lower_bound:,.2f}")
print(f"Upper Threshold       : ${upper_bound:,.2f}")

print(f"\nTotal Records         : {len(silver_df):,}")
print(f"Normal Records        : {len(silver_non_anomaly_df):,}")
print(f"Anomalous Records     : {len(silver_anomaly_df):,}")

print("\nFlagged Anomalies")
print("=" * 80)

display(silver_anomaly_display_df)


Potential Resale Price Anomalies
Lower Threshold       : $5,000.00
Upper Threshold       : $796,250.00

Total Records         : 982,011
Normal Records        : 956,747
Anomalous Records     : 25,264

Flagged Anomalies


,town,flat_type,month,floor_area_sqm,lease_commence_date,resale_price,lower_threshold,upper_threshold,anomaly_type,threshold_difference
0,BUKIT MERAH,5 ROOM,2026-04-01,113.0,2019,1728000.0,5000.0,796250.0,Above Upper Threshold,931750.0
1,QUEENSTOWN,5 ROOM,2026-02-01,122.0,2016,1700000.0,5000.0,796250.0,Above Upper Threshold,903750.0
2,QUEENSTOWN,5 ROOM,2025-06-01,122.0,2016,1658888.0,5000.0,796250.0,Above Upper Threshold,862638.0
3,QUEENSTOWN,5 ROOM,2026-06-01,122.0,2016,1650000.0,5000.0,796250.0,Above Upper Threshold,853750.0
4,BUKIT MERAH,5 ROOM,2026-03-01,112.0,2016,1648888.0,5000.0,796250.0,Above Upper Threshold,852638.0
...,...,...,...,...,...,...,...,...,...,...
25259,CHOA CHU KANG,EXECUTIVE,2024-05-01,143.0,1997,796888.0,5000.0,796250.0,Above Upper Threshold,638.0
25260,TAMPINES,4 ROOM,2026-04-01,93.0,2021,796888.0,5000.0,796250.0,Above Upper Threshold,638.0
25261,QUEENSTOWN,4 ROOM,2018-10-01,93.0,2011,796888.0,5000.0,796250.0,Above Upper Threshold,638.0
25262,QUEENSTOWN,4 ROOM,2020-11-01,83.0,2016,796888.0,5000.0,796250.0,Above Upper Threshold,638.0


### Gold 2: Data Science Team - clean resale data from January 2012 to December 2016

In [28]:
gold_data_science_df = silver_passed_df.copy()

gold_data_science_df["month"] = pd.to_datetime(
    gold_data_science_df["month"]
)

gold_data_science_df = gold_data_science_df[
    (
        gold_data_science_df["month"] >= "2012-01-01"
    )
    &
    (
        gold_data_science_df["month"] <= "2016-12-31"
    )
].copy()

gold_data_science_df["month"] = (
    gold_data_science_df["month"]
    .dt.strftime("%Y-%m")
)

print(
    f"Records for Data Science Team: "
    f"{len(gold_data_science_df):,}"
)

display(gold_data_science_df.head())

Records for Data Science Team: 92,544


,block,flat_model,flat_type,floor_area_sqm,lease_commence_date,month,remaining_lease,resale_price,storey_range,street_name,town,vault_id,dataset_id
1663,57,Terrace,3 ROOM,259.0,1972,2016-12,54,1150000.0,01 TO 03,JLN MA'MOR,KALLANG/WHAMPOA,36491,d_ea9ed51da2787afaf8e51f827c304208
2023,1G,Type S2,5 ROOM,106.0,2011,2016-09,93,1120000.0,43 TO 45,CANTONMENT RD,CENTRAL AREA,31309,d_ea9ed51da2787afaf8e51f827c304208
2333,1D,Type S2,5 ROOM,107.0,2011,2016-11,93,1100000.0,46 TO 48,CANTONMENT RD,CENTRAL AREA,34599,d_ea9ed51da2787afaf8e51f827c304208
2352,1B,Type S2,5 ROOM,106.0,2011,2016-11,93,1100000.0,31 TO 33,CANTONMENT RD,CENTRAL AREA,34598,d_ea9ed51da2787afaf8e51f827c304208
2390,8,DBSS,5 ROOM,119.0,2011,2016-08,93,1100000.0,28 TO 30,BOON KENG RD,KALLANG/WHAMPOA,30012,d_ea9ed51da2787afaf8e51f827c304208


# Output data

### Output 1: Raw dataset

In [38]:
# =============================================================================
# DATA OUTPUT 1 - RAW DATASETS
# =============================================================================

raw_output_path = Path("output/raw")
raw_output_path.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 100)
print("EXPORTING RAW DATASETS")
print("=" * 100)

total_records = 0

for dataset_id, raw_df in bronze_child_datasets.items():

    output_file = (
        raw_output_path
        / f"{dataset_id}.csv"
    )

    raw_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"\n✓ Exported {dataset_id}.csv "
        f"({len(raw_df):,} records)"
    )

    preview_columns = [
        column
        for column in [
            "_dataset_id",
            "month",
            "town",
            "flat_type",
            "block",
            "street_name",
            "resale_price"
        ]
        if column in raw_df.columns
    ]

    display(
        raw_df[preview_columns].head(3)
    )

    print("-" * 100)

print("\nEXPORT SUMMARY")
print("=" * 100)

print(f"Output Folder      : {raw_output_path.resolve()}")
print(f"Datasets Exported  : {len(bronze_child_datasets):,}")
print(f"Total Records      : {total_records:,}")

EXPORTING RAW DATASETS

✓ Exported d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv (235,808 records)


,_dataset_id,month,town,flat_type,block,street_name,resale_price
0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,232000
1,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,250000
2,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,262000


----------------------------------------------------------------------------------------------------

✓ Exported d_43f493c6c50d54243cc1eab0df142d6a.csv (369,651 records)


,_dataset_id,month,town,flat_type,block,street_name,resale_price
0,d_43f493c6c50d54243cc1eab0df142d6a,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,147000
1,d_43f493c6c50d54243cc1eab0df142d6a,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,144000
2,d_43f493c6c50d54243cc1eab0df142d6a,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,159000


----------------------------------------------------------------------------------------------------

✓ Exported d_2d5ff9ea31397b66239f245f57751537.csv (52,203 records)


...

----------------------------------------------------------------------------------------------------

✓ Exported d_ebc5ab87086db484f88045b47411ebc5.csv (287,196 records)


,_dataset_id,month,town,flat_type,block,street_name,resale_price
0,d_ebc5ab87086db484f88045b47411ebc5,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,9000
1,d_ebc5ab87086db484f88045b47411ebc5,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,6000
2,d_ebc5ab87086db484f88045b47411ebc5,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,8000


----------------------------------------------------------------------------------------------------

✓ Exported d_ea9ed51da2787afaf8e51f827c304208.csv (37,153 records)


,_dataset_id,month,town,flat_type,block,street_name,resale_price
0,d_ea9ed51da2787afaf8e51f827c304208,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,255000
1,d_ea9ed51da2787afaf8e51f827c304208,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,275000
2,d_ea9ed51da2787afaf8e51f827c304208,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,285000


----------------------------------------------------------------------------------------------------

EXPORT SUMMARY
Output Folder      : /Users/jackgoh/Desktop/Jack/HDB Senior Data Engineer Assignment/output/raw
Datasets Exported  : 5
Total Records      : 0


### Output 2: Cleaned dataset

In [41]:
# =============================================================================
# DATA OUTPUT 2 - CLEANED DATASET
# =============================================================================

cleaned_output_path = Path("output/cleaned")
cleaned_output_path.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_output_file = (
    cleaned_output_path
    / "cleaned_hdb_resale.csv"
)

silver_cleaned_df.to_csv(
    cleaned_output_file,
    index=False
)

print(
    f"✓ Exported cleaned_hdb_resale.csv "
    f"({len(silver_cleaned_df):,} records)"
)

preview_columns = [
    column
    for column in [
        "month",
        "town",
        "flat_type",
        "block",
        "street_name",
        "remaining_lease",
        "floor_area_sqm",
        "resale_price"
    ]
    if column in silver_cleaned_df.columns
]

display(
    silver_cleaned_df[
        preview_columns
    ].head(3)
)

print(
    f"\nCleaned dataset exported to: "
    f"{cleaned_output_file.resolve()}"
)

print(
    f"Number of cleaned records: "
    f"{len(silver_cleaned_df):,}"
)

✓ Exported cleaned_hdb_resale.csv (982,011 records)


,month,town,flat_type,block,street_name,remaining_lease,floor_area_sqm,resale_price
0,2026-04-01,BUKIT MERAH,5 ROOM,96A,HENDERSON RD,92 years 01 month,113.0,1728000.0
1,2026-02-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,89 years 03 months,122.0,1700000.0
2,2025-06-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,89 years 11 months,122.0,1658888.0



Cleaned dataset exported to: /Users/jackgoh/Desktop/Jack/HDB Senior Data Engineer Assignment/output/cleaned/cleaned_hdb_resale.csv
Number of cleaned records: 982,011


### Output 3: Transformed dataset

In [40]:
# =============================================================================
# DATA OUTPUT 3 - TRANSFORMED DATASET
# =============================================================================

transformed_output_path = Path("output/transformed")
transformed_output_path.mkdir(
    parents=True,
    exist_ok=True
)

transformed_output_file = (
    transformed_output_path
    / "silver_transformed_dataset.csv"
)

silver_transformed_df.to_csv(
    transformed_output_file,
    index=False
)

print(
    f"✓ Exported silver_transformed_dataset.csv "
    f"({len(silver_transformed_df):,} records)"
)

preview_columns = [
    column
    for column in [
        "month",
        "town",
        "flat_type",
        "block",
        "street_name",
        "remaining_lease",
        "resale_identifier"
    ]
    if column in silver_transformed_df.columns
]

display(
    silver_transformed_df[
        preview_columns
    ].head(3)
)

print(
    f"\nTransformed dataset exported to: "
    f"{transformed_output_file.resolve()}"
)

print(
    f"Number of transformed records: "
    f"{len(silver_transformed_df):,}"
)

✓ Exported silver_transformed_dataset.csv (982,011 records)


,month,town,flat_type,block,street_name,remaining_lease,resale_identifier
0,2026-04-01,BUKIT MERAH,5 ROOM,96A,HENDERSON RD,91 years 5 months,S0961004B
1,2026-02-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,88 years 5 months,S0921102Q
2,2025-06-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,88 years 5 months,S0921206Q



Transformed dataset exported to: /Users/jackgoh/Desktop/Jack/HDB Senior Data Engineer Assignment/output/transformed/silver_transformed_dataset.csv
Number of transformed records: 982,011


### Output 4: Failed dataset

In [42]:
# =============================================================================
# DATA OUTPUT 4 - FAILED DATASET
# =============================================================================

failed_output_path = Path("output/failed")
failed_output_path.mkdir(
    parents=True,
    exist_ok=True
)

failed_datasets = []


# =============================================================================
# DUPLICATE RECORD FAILURES
# =============================================================================

if (
    "silver_failed_duplicate_df" in globals()
    and not silver_failed_duplicate_df.empty
):

    duplicate_df = silver_failed_duplicate_df.copy()

    duplicate_df["failure_reason"] = (
        "Duplicate Composite Key"
    )

    failed_datasets.append(
        duplicate_df
    )


# =============================================================================
# GREAT EXPECTATIONS VALIDATION FAILURES
# =============================================================================

if (
    "silver_failed_validation_df" in globals()
    and not silver_failed_validation_df.empty
):

    validation_df = (
        silver_failed_validation_df.copy()
    )

    validation_df["failure_reason"] = (
        "Failed Data Validation"
    )

    failed_datasets.append(
        validation_df
    )


# =============================================================================
# COMBINE ALL FAILED RECORDS
# =============================================================================

if failed_datasets:

    failed_output_df = pd.concat(
        failed_datasets,
        ignore_index=True
    )

else:

    # Preserve a meaningful schema even when no rows failed.
    failed_output_df = pd.DataFrame(
        columns=[
            "failure_reason"
        ]
    )


# =============================================================================
# EXPORT FAILED DATASET
# =============================================================================

failed_output_file = (
    failed_output_path
    / "failed_hdb_resale.csv"
)

failed_output_df.to_csv(
    failed_output_file,
    index=False
)


# =============================================================================
# DISPLAY EXPORT RESULT
# =============================================================================

print(
    f"✓ Exported failed_hdb_resale.csv "
    f"({len(failed_output_df):,} records)"
)

if not failed_output_df.empty:

    preview_columns = [
        column
        for column in [
            "dataset_id",
            "vault_id",
            "month",
            "town",
            "flat_type",
            "block",
            "street_name",
            "resale_price",
            "failure_reason"
        ]
        if column in failed_output_df.columns
    ]

    display(
        failed_output_df[
            preview_columns
        ].head(3)
    )

else:

    print(
        "\nNo failed records were produced by the pipeline."
    )

print(
    f"\nFailed dataset exported to: "
    f"{failed_output_file.resolve()}"
)

print(
    f"Number of failed records: "
    f"{len(failed_output_df):,}"
)

✓ Exported failed_hdb_resale.csv (0 records)

No failed records were produced by the pipeline.

Failed dataset exported to: /Users/jackgoh/Desktop/Jack/HDB Senior Data Engineer Assignment/output/failed/failed_hdb_resale.csv
Number of failed records: 0


### Output 5: Hashed dataset

In [43]:
# =============================================================================
# DATA OUTPUT 5 - HASHED DATASET
# =============================================================================

hashed_output_path = Path("output/hashed")
hashed_output_path.mkdir(
    parents=True,
    exist_ok=True
)

hashed_output_file = (
    hashed_output_path
    / "hashed_hdb_resale.csv"
)

silver_hashed_output_df.to_csv(
    hashed_output_file,
    index=False
)

print(
    f"✓ Exported hashed_hdb_resale.csv "
    f"({len(silver_hashed_output_df):,} records)"
)

preview_columns = [
    column
    for column in [
        "month",
        "town",
        "flat_type",
        "block",
        "street_name",
        "remaining_lease",
        "hashed_resale_identifier"
    ]
    if column in silver_hashed_output_df.columns
]

display(
    silver_hashed_output_df[
        preview_columns
    ].head(3)
)

print(
    f"\nHashed dataset exported to: "
    f"{hashed_output_file.resolve()}"
)

print(
    f"Number of hashed records: "
    f"{len(silver_hashed_output_df):,}"
)

✓ Exported hashed_hdb_resale.csv (982,011 records)


,month,town,flat_type,block,street_name,remaining_lease,hashed_resale_identifier
0,2026-04-01,BUKIT MERAH,5 ROOM,96A,HENDERSON RD,91 years 5 months,acf02627fe56dc5b7fb5dd8656b2165840840e43fcfa6d...
1,2026-02-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,88 years 5 months,fe8c51e42b8e500e229d8b880a01c0db4aff1fe768a8fe...
2,2025-06-01,QUEENSTOWN,5 ROOM,92,DAWSON RD,88 years 5 months,b402eb5dc0d00f534459113590b8eb519ad7aba78a6d64...



Hashed dataset exported to: /Users/jackgoh/Desktop/Jack/HDB Senior Data Engineer Assignment/output/hashed/hashed_hdb_resale.csv
Number of hashed records: 982,011
